# 08 — Confrontare riferimento, albero pesato e Random Forest

T14. Nessun nuovo fit: leggiamo gli artefatti dei notebook 04/06/07 e verifichiamo
che tutti i modelli siano valutati sugli stessi record, risposte e seed.
Giudichiamo insieme recupero degli attacchi e falsi positivi, come concordato nel grill-me.


## Esito misurato sulle cinque prove concordate

Stessi seed 1–5 e stessi record di validation per i tre modelli. Le medie delle
metriche sono calcolate per run; DS è la deviazione standard campionaria fra le cinque prove.

| Modello | Precision U2R media | Recall U2R medio | FP U2R medi/run | Macro F1 media ± DS |
|---|---:|---:|---:|---:|
| Albero15 senza pesi | 49.62% | 40.00% | 3.2 | 0.875387 ± 0.055038 |
| Albero15 balanced | 24.42% | 58.00% | 24.2 | 0.810615 ± 0.037510 |
| Random Forest balanced | 78.34% | 60.00% | 2.2 | 0.920033 ± 0.031215 |

L'albero pesato aumenta il recall U2R, ma i falsi positivi medi salgono da 3,2 a 24,2
per run e la precision scende al 24,42%. Il macro F1 diminuisce: su queste prove il
compromesso è sfavorevole rispetto al riferimento.

La Random Forest migliora contemporaneamente precision e recall U2R medi e riduce
i falsi positivi medi a 2,2. Il macro F1 sale a 0,920033; è il candidato più promettente
di questo confronto. **U2R resta difficile:** errore medio 40%, con singole prove dal20%
al70%; DS del recall 21,21 punti percentuali. Non è una promozione automatica a modello finale.

Per R2L la Random Forest raggiunge recall medio 97,09% e precision 94,83%, contro
93,47%/95,98% del riferimento: recupera più casi, con un piccolo calo della precision.
L'albero pesato ha recall 96,28% ma precision 60,02%, con137,6 falsi positivi medi/run.

Le cinque validation contengono 31 U2R distinti riutilizzati fra run. Non sono campioni
indipendenti; non attribuiamo significatività statistica alle sole medie e non misuriamo
prestazioni sul test ufficiale. La pesatura e l'ensemble sono varianti di sviluppo nostre.


In [ ]:
from pathlib import Path
import sys
import json
import hashlib

import numpy as np
import pandas as pd
from IPython.display import display

project_root = Path.cwd() if (Path.cwd() / 'src').is_dir() else Path.cwd().parent
sys.path.insert(0, str(project_root))
from src.data import COLUMN_NAMES, FEATURE_COLUMNS, CATEGORICAL_COLUMNS
from src.targets import ATTACK_TO_FAMILY, FAMILY_ORDER


## 1. Caricare i tre risultati

Il riferimento è l'albero15 senza pesi del notebook 04. Ogni nome identifica una
configurazione, non una nuova suddivisione dei dati. I file originali restano invariati.


In [ ]:
sources = {
    'decision_tree': project_root / 'artifacts/rare_class_reference',
    'decision_tree_balanced': project_root / 'artifacts/decision_tree_balanced',
    'random_forest_balanced': project_root / 'artifacts/random_forest_balanced',
}
reference_id = 'decision_tree'
experiments = {}
arrays_by_model = {}
for model_id, folder in sources.items():
    experiments[model_id] = json.loads((folder / 'results.json').read_text(encoding='utf-8'))
    with np.load(folder / 'predictions.npz', allow_pickle=False) as archive:
        arrays_by_model[model_id] = {key: archive[key].copy() for key in archive.files}


## 2. Prima abbinare, poi confrontare

La stessa dimensione della validation non basta. Controlliamo sorgente, seed, ordine
delle famiglie, indici e risposte vere, run per run. Se una verifica fallisce,
il confronto si interrompe invece di produrre medie fra osservazioni diverse.


In [ ]:
reference = experiments[reference_id]
for model_id, experiment in experiments.items():
    assert experiment['source_sha256'] == reference['source_sha256']
    assert experiment['seeds'] == reference['seeds']
    assert experiment['feature_count'] == reference['feature_count']
    assert experiment['family_order'] == reference['family_order']
    for run, seed in enumerate(reference['seeds'], start=1):
        for suffix in ['validation_indices', 'truth']:
            key = f'run_{run:02d}_{suffix}'
            np.testing.assert_array_equal(arrays_by_model[model_id][key], arrays_by_model[reference_id][key])


## 3. Medie e dispersione per tutte le famiglie

Manteniamo le sole righe profondità15 del riferimento e aggiungiamo l'identità del
modello in memoria. std è la deviazione standard campionaria fra cinque run sovrapposte,
non un intervallo di confidenza. Le metriche vengono mediate per run.


In [ ]:
run_frames, class_frames = [], []
for model_id, experiment in experiments.items():
    runs = pd.DataFrame(experiment['run_metrics'])
    classes = pd.DataFrame(experiment['class_metrics'])
    run_frames.append(runs.loc[runs['max_depth'].eq(15)].assign(model_id=model_id))
    class_frames.append(classes.loc[classes['max_depth'].eq(15)].assign(model_id=model_id))
run_results = pd.concat(run_frames, ignore_index=True)
class_results = pd.concat(class_frames, ignore_index=True)
overall_summary = run_results.groupby('model_id', sort=False)[
    ['accuracy', 'macro_f1', 'weighted_f1']
].agg(['mean', 'std', 'min', 'max'])
class_summary = class_results.groupby(['model_id', 'famiglia'], sort=False)[
    ['precision', 'recall', 'f1', 'quota_errori', 'falsi_positivi']
].agg(['mean', 'std', 'min', 'max'])
display(overall_summary.round(6))
display(class_summary.loc[(slice(None), ['R2L', 'U2R']), :].round(4))
assert len(run_results) == len(sources) * len(reference['seeds']) and len(class_results) == len(run_results) * len(FAMILY_ORDER)


## 4. Matrici normalizzate medie con la stessa scala

La diagonale indica il recall medio per famiglia vera. I falsi positivi vengono
letti anche con precision e conteggi: percentuali piccole nelle famiglie numerose
possono pesare molto sulle previsioni di una classe rara.


In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

names = {'decision_tree': 'Albero15 senza pesi',
         'decision_tree_balanced': 'Albero15 balanced',
         'random_forest_balanced': 'Random Forest balanced'}
fig, axes = plt.subplots(1, 3, figsize=(16, 5), constrained_layout=True)
for ax, (model_id, experiment) in zip(axes, experiments.items()):
    matrix = np.asarray(experiment['normalized_confusion_mean']['15'])
    ConfusionMatrixDisplay(matrix, display_labels=FAMILY_ORDER).plot(
        values_format='.1%', cmap='Blues', colorbar=False, im_kw={'vmin': 0, 'vmax': 1}, ax=ax,
    )
    ax.set(title=names[model_id], xlabel='Famiglia prevista', ylabel='Famiglia vera')
fig.suptitle('Stessi cinque split: media delle confusion matrix normalizzate')
plt.show()


## 5. Differenze abbinate per seed

Sottraiamo il riferimento dalla variante sulla stessa run. Un delta positivo
di recall significa più attacchi riconosciuti; un delta negativo di precision
segnala più rumore nelle previsioni della famiglia. Nessuna di queste differenze,
da sola, costituisce una scelta automatica del modello finale.


In [ ]:
paired_frames = []
metric_columns = ['precision', 'recall', 'f1', 'quota_errori', 'falsi_positivi']
base = class_results.loc[class_results['model_id'].eq(reference_id)].set_index(['seed', 'famiglia'])
for model_id in sources:
    if model_id == reference_id:
        continue
    candidate = class_results.loc[class_results['model_id'].eq(model_id)].set_index(['seed', 'famiglia'])
    pd.testing.assert_index_equal(base.sort_index().index, candidate.sort_index().index)
    delta = candidate[metric_columns] - base[metric_columns]
    paired_frames.append(delta.add_prefix('delta_').reset_index().assign(model_id=model_id))
paired_deltas = pd.concat(paired_frames, ignore_index=True)
display(paired_deltas.loc[paired_deltas['famiglia'].isin(['R2L', 'U2R'])].groupby(
    ['model_id', 'famiglia'], sort=False,
).agg({f'delta_{metric}': ['mean', 'std', 'min', 'max'] for metric in metric_columns}).round(4))


## 6. Salvare il confronto e fermare gli esperimenti

Le varianti peggiori restano nel confronto. Nessun altro peso, profondità o modello
viene provato dopo aver visto questi esiti. I record U2R distinti osservati rimangono
gli stessi; le ripetizioni non creano dati nuovi. Il test ufficiale resta separato.


In [ ]:
artifacts = project_root / 'artifacts/rare_class_comparison'
artifacts.mkdir(parents=True, exist_ok=True)
run_results.to_csv(artifacts / 'run_metrics.csv', index=False)
class_results.to_csv(artifacts / 'class_metrics.csv', index=False)
overall_summary.to_csv(artifacts / 'overall_summary.csv')
class_summary.to_csv(artifacts / 'class_summary.csv')
paired_deltas.to_csv(artifacts / 'paired_deltas.csv', index=False)
fig.savefig(artifacts / 'mean_confusion_matrices.png', dpi=160)
print('Confronto salvato. Interpretazione in docs/RARE_CLASS_COMPARISON.md.')


## Interpretazione e limiti

Criterio concordato: leggere recall e precision insieme; un recupero maggiore con
molti più falsi positivi è un compromesso da discutere. Il macro F1 riassume tutte
le famiglie, ma non sostituisce le loro misure. Le prove restano di sviluppo sulla
stessa fonte e su split sovrapposti: nessuna conclusione di generalizzazione.
La scelta successiva richiede la revisione dell'utente.
[Report del blocco](../docs/RARE_CLASS_COMPARISON.md).
